# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hacker3code/Flyrank-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Lane 2 — Refresh / Content Opportunity Scoring

**ML task type: Ranking**

The goal is to rank pages by how strongly they should be considered for human review. This is a ranking problem because the practical output is not simply "yes" or "no" for every page. A reviewer has limited time, so the useful output is an ordered list of pages to examine first.

The ranking can use signals such as impressions, clicks, CTR, average position, content age, freshness, sessions, engagement, and other observable page-level signals.

The output supports a content decision: which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring.

In [5]:
# This cell is for CODE (numbers, a query, a check).
print("Task type: Ranking")
print("Unit of decision: content page")
print("Output: ranked review queue")

Task type: Ranking
Unit of decision: content page
Output: ranked review queue


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

url = "https://raw.githubusercontent.com/hacker3code/Flyrank-AI/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("Rows:", len(df))
print("Declining proxy labels:", df["is_declining_label"].sum())
print(
    "Declining percentage:",
    f"{df['is_declining_label'].mean() * 100:.2f}%"
)

Rows: 30000
Declining proxy labels: 16262
Declining percentage: 54.21%


### Target / Proxy

For the starter exercise, I will use `is_declining_label` as a proxy target:

`is_declining_label = (trend_direction == "down")`

This proxy represents whether a page is currently showing a downward trend according to the starter dataset.

The ideal target would be stronger and future-looking: whether a page's performance declines or recovers in a future time window after the decision point. For example, features from a prior 90-day window could be used to predict decline during the following 30 days.

I am therefore treating `is_declining_label` as a bootstrap proxy for this assignment, not as proof that refreshing a page will improve its performance.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric: Precision@50

My primary success metric is Precision@50.

Precision@50 measures the proportion of the top 50 pages in the ranked review queue that are positive according to the evaluation target.

This matches the real decision because a content team has limited review capacity. I care more about making the first 50 recommendations useful than about getting every page classified correctly.

For the starter exercise, Precision@50 will be measured against the `is_declining_label` proxy. This measures how well the ranking identifies pages with the starter decline signal; it does not prove that those pages would benefit from a refresh.

In [8]:
# This cell is for CODE (numbers, a query, a check).
print("Primary success metric: Precision@50")
print("Reason: it matches a limited review capacity of 50 pages.")


Primary success metric: Precision@50
Reason: it matches a limited review capacity of 50 pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of analysis

**One row = one content page.**

Each row represents an anonymized content item and contains observed signals associated with that page, such as its 90-day impressions, clicks, sessions, CTR, position, age, freshness, and trend.

The decision is made at the page level because the eventual output is a ranked list of pages for a content reviewer to inspect.

In [9]:
# This cell is for CODE (numbers, a query, a check).
page_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "trend_direction",
    "is_declining_label",
]

page_df = df[page_cols].copy()

print("One row represents one content page.")
print("Rows:", len(page_df))
print("Columns:", len(page_df.columns))

display(page_df.head(10))


One row represents one content page.
Rows: 30000
Columns: 10


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,17,0.76,10.6,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,0.05,20.3,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,78,0.49,6.2,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,0.13,44.0,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,0.03,8.5,147,down,1
6,content_9a34b442b552,client_8722616204,20,0,1,0.00,7.0,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,28,0.06,21.2,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,68,0.09,46.0,90,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,3,0.16,4.9,257,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML may beat a fixed rule

A fixed rule is useful as a transparent baseline, but a single rule may not capture the different combinations of signals that make a page worth reviewing.

For example, a page can have high impressions but low CTR, or strong position but weak engagement, or good visibility combined with old content and a declining trend. The importance of these signals may also differ across pages.

An ML model can learn combinations and nonlinear relationships between multiple observable signals instead of relying on one manually chosen threshold.

However, ML is not automatically better. I will compare it against a transparent baseline and keep the more complex approach only if it produces a meaningful improvement on a decision-relevant metric such as Precision@50.

The final output remains decision-support: the model ranks pages for human review rather than automatically deciding that a page must be refreshed.

In [10]:
# This cell is for CODE (numbers, a query, a check).
print("Baseline: transparent rules")
print("Candidate ML approach: learned ranking/classification model")
print("Comparison metric: Precision@50")
print("Final use: decision-support for human review")


Baseline: transparent rules
Candidate ML approach: learned ranking/classification model
Comparison metric: Precision@50
Final use: decision-support for human review


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.